# Lite-Mono 학습 (Colab)

KITTI 주행 사진으로 Lite-Mono(사진 한 장 → 깊이)를 학습하고 채점한다. 각 단계와 파일의 자세한 설명은 저장소의 [GUIDE.md](https://github.com/Sangbum-K/Lite-Mono-Depth-Estimation/blob/Colab/GUIDE.md)에 있다.

**시작 전 확인**
- Drive `내 드라이브/lite_mono/data/`에 파일 4개: `kitti_jpg.tar`, `eigen_gt_depths.npz`, `test_gt.npz`, `val_gt.npz` (PC에서 `scripts/download_kitti_pc.sh`로 만든 것)
- 메뉴 **런타임 → 런타임 유형 변경 → GPU** (L4)

**실행 순서**

| 상황 | 실행할 단계 |
|---|---|
| 맨 처음 | 1 → 2 → 3 → 4 → 5 |
| 학습 시작, 또는 끊긴 학습 이어하기 | 1 → 2 → 4 → 6 |
| 학습이 끝난 뒤 | 1 → 2 → 4 → 7 |

- `/content`(코드·사진)는 세션이 끝나면 지워진다. 남겨야 하는 것은 모두 Drive `lite_mono/`에 저장된다.
- 세션이 끊기면 위 표대로 1단계부터 다시 실행한다. 학습은 마지막으로 끝난 epoch 다음부터 이어간다.

## 1. 준비 (매 세션)
Drive 연결 → 설정 → 코드 받기(이미 있으면 GitHub 최신으로 업데이트) → 패키지 설치 → 환경 기록.

- 처음에는 Google 로그인 팝업이 뜬다. **Drive와 같은 계정**을 고르고 **권한 체크박스를 모두 선택**한다. 하나라도 빼면 `mount failed`가 난다.
- `RUN_ID`는 학습 이름이다. 학습을 처음 시작할 때 정하고 끝날 때까지 바꾸지 않는다. 같은 이름이어야 이어서 학습한다.
- 정상이면 마지막에 GPU 이름과 `준비 완료`가 나온다.

In [ ]:
import os
from datetime import datetime
from google.colab import drive
drive.mount('/content/drive')

# 경로 (바꿀 일 없음)
DRIVE = '/content/drive/MyDrive/lite_mono'    # 영구 저장: 데이터, 가중치, 학습 결과
REPO = '/content/Lite-Mono-Depth-Estimation'  # 코드. 세션마다 GitHub에서 받음
KITTI = '/content/kitti_data'                 # 사진. 4단계에서 Drive의 tar를 풀어 만듦
REPO_URL = 'https://github.com/Sangbum-K/Lite-Mono-Depth-Estimation.git'
REPO_BRANCH = 'Colab'

# 학습 설정 (6·7단계)
RUN_ID = '20261109_baseline_s42'  # 날짜_방법_seed. 한 번 정하면 그 학습이 끝날 때까지 고정
SEED, NUM_WORKERS = 42, 8         # 서로 비교할 학습끼리 같게 둔다

os.environ.update(DRIVE=DRIVE, REPO=REPO, KITTI=KITTI, REPO_URL=REPO_URL, REPO_BRANCH=REPO_BRANCH,
                  RUN_ID=RUN_ID, SEED=str(SEED), NUM_WORKERS=str(NUM_WORKERS))

missing = [f for f in ['kitti_jpg.tar', 'eigen_gt_depths.npz', 'test_gt.npz', 'val_gt.npz']
           if not os.path.isfile(f'{DRIVE}/data/{f}')]
assert not missing, f'{DRIVE}/data 에 없는 파일: {missing} (Drive에서 lite_mono 폴더가 내 드라이브 바로 아래에 있는지 확인)'

# 코드: 없으면 받고, 있으면 최신으로 업데이트
!if [ -d $REPO ]; then git -C $REPO pull -q --ff-only; else git clone -q -b $REPO_BRANCH $REPO_URL $REPO; fi
assert os.path.isdir(f'{REPO}/vendor/Lite-Mono/datasets'), '코드 받기 실패: 위 git 오류를 확인'
!git --no-pager -C $REPO log -1 --format='코드 버전: %h %s'
!pip install -q -r $REPO/environment/requirements-colab.txt

# 이번 세션의 GPU·패키지 버전 기록 -> Drive lite_mono/env/
os.makedirs(f'{DRIVE}/env', exist_ok=True)
log = f'{DRIVE}/env/session_{datetime.now():%Y%m%d_%H%M}.txt'
!(date; git --no-pager -C $REPO log -1 --format=%H; nvidia-smi; nproc; free -g; df -h /content; python -V; pip freeze) > "{log}" 2>&1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'GPU 없음: 런타임 유형을 GPU로 바꿀 것'
print('준비 완료. 학습 이름:', RUN_ID)

## 2. 공식 가중치 (매 세션)
Lite-Mono 저자가 공개한 가중치를 Drive `lite_mono/pretrained/`에 받는다. 처음 한 번만 받고, 그다음부터는 파일이 손상되지 않았는지만 확인한다.

- `lite-mono-pretrain.pth`: ImageNet으로 미리 학습한 인코더. 6단계 학습의 출발점
- `lite-mono_640x192/`: 저자가 KITTI 학습을 끝낸 모델. 5단계 점검용

정상이면 세 줄 모두 `OK`.

In [ ]:
%%bash
set -e
W=$DRIVE/pretrained; mkdir -p $W
[ -f $W/lite-mono-pretrain.pth ] || curl -fsSL -o $W/lite-mono-pretrain.pth https://surfdrive.surf.nl/files/index.php/s/InMMGd5ZP2fXuia/download
[ -f $W/lite-mono_640x192/encoder.pth ] || { curl -fsSL -o /tmp/lm.zip https://surfdrive.surf.nl/files/index.php/s/CUjiK221EFLyXDY/download && unzip -o -q /tmp/lm.zip -d $W; }
cd $W && sha256sum -c <<EOF
9877014dfaede372ca887dea354cb534788eb68b6ded13a9a1d39c715c04fc9f  lite-mono-pretrain.pth
9c5bc031b4a6a8d9c6df5f525c6740594b7d814841cc924f61565d0da624e046  lite-mono_640x192/encoder.pth
c452dd45e23a4cddd823943a43e793537c0095c0c8fc2037a8a4e6ad30acc224  lite-mono_640x192/depth.pth
EOF

## 3. 코드 점검 (처음 한 번, 그리고 코드를 바꾼 뒤)
데이터 없이 몇 초 만에 확인한다: 코드와 가중치가 맞는지, 파라미터 수(약 3.1M), 호환 수정, GPU에서 학습 한 스텝.

마지막 줄이 `smoke test passed`면 통과. 맨 위의 timm `FutureWarning`은 무시해도 된다.

In [ ]:
!cd $REPO && python scripts/smoke_test.py --pretrained $DRIVE/pretrained

## 4. 데이터 풀기 (매 세션)
Drive의 `kitti_jpg.tar`(6.3GB)를 Colab 로컬 디스크 `/content/kitti_data`에 푼다. Drive에서 사진 수만 장을 바로 읽으면 학습이 느려지기 때문이다. 공식 test 정답(`eigen_gt_depths.npz`)은 공식 채점 코드가 읽는 위치에 복사한다.

마지막 줄이 `check: RGB 45259/45259 present, LiDAR 0/0 present, calib missing 0`이면 정상. 이미 풀려 있으면 압축 풀기는 건너뛴다.

In [ ]:
%%bash
set -e
[ -d $KITTI/2011_09_26 ] || tar -xf $DRIVE/data/kitti_jpg.tar -C /content
cp $DRIVE/data/eigen_gt_depths.npz $REPO/vendor/Lite-Mono/splits/eigen/gt_depths.npz
cd $REPO && python scripts/prepare_kitti.py --kitti_out $KITTI  # 파일 점검만 (다운로드 없음)

## 5. 저자 모델 채점 (처음 한 번)
저자가 학습을 끝낸 모델로 test 697장을 채점한다. 데이터와 채점 코드가 맞는지 확인하는 단계다.

- 첫 번째 명령: 공식 채점 코드(`evaluate_depth.py`)
- 두 번째 명령: 이미지별 채점 코드(`analysis/eval_per_image.py`). 이미지마다 점수를 `pretrained/lite-mono_640x192/per_image_test_files.csv`에 저장

두 결과 모두 `0.107 0.765 4.561 0.183 0.886 0.963 0.983`(abs_rel ~ a3) 근처여야 한다. 다르면 학습하기 전에 원인부터 찾는다.

In [ ]:
!cd $REPO/vendor/Lite-Mono && python evaluate_depth.py --load_weights_folder $DRIVE/pretrained/lite-mono_640x192 --data_path $KITTI --model lite-mono
!cd $REPO && python analysis/eval_per_image.py eval --split_file data/splits/test_files.txt --gt $DRIVE/data/test_gt.npz --data_path $KITTI --weights $DRIVE/pretrained/lite-mono_640x192

## 6. 학습 (끊기면 1 → 2 → 4 → 6으로 이어서)
논문과 같은 설정: ImageNet 인코더에서 시작, 학습 목록 39,810개(eigen_zhou), 640×192, batch 12, 30 epoch, lr `0.0001 5e-6 31 0.0001 1e-5 31`. `options.py`의 기본값(batch 16, 50 epoch)이 논문과 달라서 모두 직접 지정한다.

- 처음 실행하면 ImageNet 인코더에서 시작하고, 다시 실행하면 Drive에 저장된 마지막 epoch 다음부터 이어간다.
- 결과는 Drive `lite_mono/outputs/<RUN_ID>/`에 쌓인다: `models/weights_0` ~ `weights_29`(epoch마다), `train_log.txt`, `gpu_log.csv`, `sessions.txt`, TensorBoard 기록.
- 로그 한 줄 예: `epoch 3 | 12.5% | ... | loss: 0.12 | time elapsed: 01h30m | time left: 10h20m`. `%`는 전체 30 epoch 중 진행률이다. loss는 배치마다 흔들리므로 전체적으로 내려가는지만 본다.
- 학습 셀이 도는 동안에는 다른 셀을 실행할 수 없다. 그래프를 보려면 TensorBoard 셀을 **학습 셀보다 먼저** 실행해 둔다.

**세션 유지 (학습 셀 실행 직전, 브라우저에서 한 번):** Colab 탭에서 F12 → Console 탭에 아래를 붙여넣는다(붙여넣기가 막히면 `allow pasting` 입력 후 Enter). 60초마다 연결 버튼을 눌러 유휴 판정을 막는다. 콘솔에 `KeepAlive`가 1분마다 찍히면 동작 중이고, `button not found`가 찍히면 Colab 화면 구조가 바뀐 것이다. 탭이나 PC를 끄거나 PC가 절전 모드로 들어가면 효과가 없다. 멈출 때는 `clearInterval(keepAlive)`.
```js
var keepAlive = setInterval(() => {
  const b = document.querySelector('colab-toolbar-button#connect')
         || document.querySelector('colab-connect-button')?.shadowRoot?.querySelector('#connect');
  if (b) { b.click(); console.log('KeepAlive', new Date().toLocaleTimeString()); }
  else console.log('KeepAlive: button not found');
}, 60000);
```
세션 최대 시간(Pro 약 12시간)은 이것으로 늘어나지 않는다. 학습 셀은 `&`로 백그라운드에 보내지 않는다. 셀이 끝난 것처럼 보여 유휴로 판정되기 쉽고, 로그는 이미 Drive의 `train_log.txt`에 저장된다.

In [ ]:
# (선택) 학습 그래프. 학습 셀보다 먼저 실행해 두면 학습 중에도 TensorBoard 새로고침 버튼으로 갱신된다
%load_ext tensorboard
%tensorboard --logdir $DRIVE/outputs

In [ ]:
%%shell
cd $REPO/vendor/Lite-Mono
RUN=$DRIVE/outputs/$RUN_ID; mkdir -p $RUN
LAST=$(ls $RUN/models/weights_*/train_state.json 2>/dev/null | sort -V | tail -1 | xargs -r dirname)
if [ -n "$LAST" ]; then INIT="--load_weights_folder $LAST"; else INIT="--mypretrain $DRIVE/pretrained/lite-mono-pretrain.pth"; fi
echo "$(date) start: $INIT" | tee -a $RUN/sessions.txt
nohup nvidia-smi --query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu --format=csv,noheader -l 60 >> $RUN/gpu_log.csv 2>&1 &
SMI=$!
python -u train.py --data_path $KITTI --log_dir $DRIVE/outputs --model_name $RUN_ID --model lite-mono --split eigen_zhou \
  --height 192 --width 640 --batch_size 12 --num_epochs 30 --lr 0.0001 5e-6 31 0.0001 1e-5 31 \
  --seed $SEED --num_workers $NUM_WORKERS $INIT 2>&1 | tee -a $RUN/train_log.txt
kill $SMI
echo "$(date) end" | tee -a $RUN/sessions.txt

## 7. epoch 고르기 → 최종 test 채점 (학습이 끝난 뒤)
- **7-1**: 30개 epoch 모델을 모두 val 4,424장으로 채점한다. 각 `weights_N` 폴더에 `per_image_val_files.csv`, 전체 요약은 `outputs/<RUN_ID>/val_summary.txt`.
- **7-2**: 고른 epoch를 `CHOSEN_EPOCH`에 적고 test 697장으로 채점한다. 이 숫자가 보고서에 쓰는 최종 결과다.

고르는 규칙(예: val abs_rel 최저)은 결과를 보기 전에 PLAN.md에 정해 둔다. test 점수를 보고 고르지 않는다. 주의: val 대부분은 train과 같은 순간을 반대쪽 카메라로 찍은 사진이라 val 점수는 실제보다 좋게 나올 수 있다.

In [ ]:
%%shell
cd $REPO
python -u analysis/eval_per_image.py eval --split_file data/splits/val_files.txt --gt $DRIVE/data/val_gt.npz --data_path $KITTI \
  --weights $(ls -d $DRIVE/outputs/$RUN_ID/models/weights_* | sort -V) | tee $DRIVE/outputs/$RUN_ID/val_summary.txt

In [ ]:
CHOSEN_EPOCH = 'weights_29'  # 미리 정한 규칙으로 고른 epoch
os.environ['CHOSEN'] = f"{DRIVE}/outputs/{RUN_ID}/models/{CHOSEN_EPOCH}"
!cd $REPO/vendor/Lite-Mono && python -u evaluate_depth.py --load_weights_folder $CHOSEN --data_path $KITTI --model lite-mono | tee $CHOSEN/eval_test_official.txt
!cd $REPO && python analysis/eval_per_image.py eval --split_file data/splits/test_files.txt --gt $DRIVE/data/test_gt.npz --data_path $KITTI --weights $CHOSEN